# Vectorless Reasoning-Based RAG — a from-scratch walkthrough

This notebook builds a small, self-contained version of *vectorless RAG*: retrieval-augmented generation that uses an LLM to **reason over the natural structure of a document** rather than searching a vector database of embeddings.

**The four steps:**

1. Load a PDF and split it by **page** (no chunking, no embeddings).
2. Build a tiny **tree-of-contents** — one short LLM-generated summary per page.
3. **Retrieval** = give the ToC to an LLM and let it *reason* about which pages to read.
4. **Answer** by reading only those pages, with page-number citations.

Sample document: **NIST AI 600-1 — Artificial Intelligence Risk Management Framework: Generative AI Profile**. Public domain, on-topic for GenAI, very structured (numbered sections, suggested actions, page-aligned content). Already included at `data/nist_ai_600-1.pdf`.

## 0. Setup

```bash
pip install -r requirements.txt
export ANTHROPIC_API_KEY=sk-ant-...
```

Or drop the key into a `.env` file in this folder.

In [ ]:
import os, json, pathlib
from dotenv import load_dotenv
load_dotenv()

import anthropic
from pypdf import PdfReader

PDF_PATH    = "data/nist_ai_600-1.pdf"
TOC_PATH    = "data/toc.json"            # cache for the per-page summaries
INDEX_MODEL = "claude-haiku-4-5"         # cheap, fast — used once per page at indexing time
ANSWER_MODEL = "claude-sonnet-4-6"       # smarter model for retrieval reasoning + final answer

client = anthropic.Anthropic()
print("Anthropic SDK:", anthropic.__version__)

## 1. Load the PDF, page by page

No chunking, no overlap, no embeddings. We just read each page's text and keep its page number — that's our atomic unit of retrieval.

In [ ]:
def load_pages(pdf_path: str) -> list[dict]:
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append({"page": i, "text": text})
    return pages

pages = load_pages(PDF_PATH)
print(f"Loaded {len(pages)} non-empty pages")
print(f"\n--- Sample (page {pages[10]['page']}) ---")
print(pages[10]["text"][:500], "...")

## 2. Build a tree-of-contents (one summary per page)

For each page, ask a small model to extract:

- a one-line **title**
- a 2-3 sentence **summary** of what's on the page
- a list of **keywords** / entities

This becomes our index. The whole index is small enough to fit in a single prompt during retrieval — which is the trick that makes vectorless RAG work.

We cache the result to `data/toc.json` so we only pay for this once.

In [ ]:
SUMMARY_PROMPT = """You are indexing a single page of a document for later retrieval.

Read the page text below and return a JSON object with these fields:
- "title":   a short (<= 10 words) title describing what this page is about
- "summary": 2-3 sentences capturing the main content; mention specific topics, sections, or entities
- "keywords": a list of 5-10 distinctive keywords or phrases someone might search for to find this page

Return ONLY the JSON, no preamble.

PAGE TEXT:
\"\"\"
{text}
\"\"\"
"""

def summarize_page(page: dict) -> dict:
    msg = client.messages.create(
        model=INDEX_MODEL,
        max_tokens=400,
        messages=[{"role": "user", "content": SUMMARY_PROMPT.format(text=page["text"][:6000])}],
    )
    raw = msg.content[0].text.strip()
    # Strip code fences if the model added them
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[1].rsplit("\n", 1)[0]
        if raw.startswith("json"):
            raw = raw[4:].lstrip()
    data = json.loads(raw)
    data["page"] = page["page"]
    return data

In [ ]:
def build_toc(pages: list[dict], cache_path: str) -> list[dict]:
    cache = pathlib.Path(cache_path)
    if cache.exists():
        print(f"Loading cached ToC from {cache_path}")
        return json.loads(cache.read_text())

    toc = []
    for p in pages:
        print(f"  indexing page {p['page']}/{len(pages)}...", end="\r")
        try:
            toc.append(summarize_page(p))
        except Exception as e:
            print(f"\n  page {p['page']} failed: {e}")
            toc.append({"page": p["page"], "title": "(indexing failed)", "summary": "", "keywords": []})
    cache.write_text(json.dumps(toc, indent=2))
    print(f"\nSaved ToC to {cache_path}")
    return toc

toc = build_toc(pages, TOC_PATH)
print(f"\nIndexed {len(toc)} pages. Sample entries:\n")
for entry in toc[:3]:
    print(json.dumps(entry, indent=2))
    print("---")

## 3. Reasoning-based retrieval

Given a user question, we hand the **entire ToC** to a stronger model and ask it to pick the pages most likely to contain the answer — and explain *why*. No cosine similarity, no top-k. Just reasoning.

In [ ]:
RETRIEVAL_PROMPT = """You are a retrieval agent for a document. Below is a table of contents where each entry corresponds to a single page.

Your job: choose the pages most likely to contain information needed to answer the user's question. Reason about which sections are relevant.

Return a JSON object with:
- "reasoning": a short paragraph explaining which sections of the document are relevant and why
- "pages": a list of page numbers to read (typically 1-5 pages; only include pages that genuinely help)

Return ONLY the JSON.

USER QUESTION:
{question}

TABLE OF CONTENTS:
{toc}
"""

def retrieve_pages(question: str, toc: list[dict], answer_model: str = ANSWER_MODEL) -> dict:
    # Compact ToC representation so it fits comfortably in the prompt
    toc_text = "\n".join(
        f"- p.{e['page']}: {e['title']} — {e['summary']}" for e in toc
    )
    msg = client.messages.create(
        model=answer_model,
        max_tokens=600,
        messages=[{"role": "user", "content": RETRIEVAL_PROMPT.format(question=question, toc=toc_text)}],
    )
    raw = msg.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[1].rsplit("\n", 1)[0]
        if raw.startswith("json"):
            raw = raw[4:].lstrip()
    return json.loads(raw)

## 4. Answer synthesis

Pull the full text of the chosen pages and send it to the model with the original question. The answer must cite the pages it used — that's the explainability win over vector RAG.

In [ ]:
ANSWER_PROMPT = """You are answering a user's question using ONLY the document pages provided below. If the pages do not contain enough information, say so honestly.

Cite the page numbers you used inline, like (p. 12). Be concise and specific.

USER QUESTION:
{question}

PAGES:
{pages}
"""

def ask(question: str, toc: list[dict], pages: list[dict], verbose: bool = True) -> str:
    selection = retrieve_pages(question, toc)
    chosen = selection["pages"]
    page_lookup = {p["page"]: p["text"] for p in pages}
    pages_text = "\n\n".join(
        f"=== Page {n} ===\n{page_lookup.get(n, '(page not found)')}" for n in chosen
    )

    if verbose:
        print("Retrieval reasoning:")
        print(" ", selection["reasoning"])
        print("Chosen pages:", chosen)
        print()

    msg = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=800,
        messages=[{"role": "user", "content": ANSWER_PROMPT.format(question=question, pages=pages_text)}],
    )
    return msg.content[0].text.strip()

## 5. Try it

In [ ]:
question = "What are some suggested actions for governing risks specific to generative AI?"
print(ask(question, toc, pages))

In [ ]:
question = "Which risks does NIST identify as unique to or exacerbated by generative AI compared to traditional AI?"
print(ask(question, toc, pages))

In [ ]:
question = "How does the document define 'confabulation' and why is it a risk?"
print(ask(question, toc, pages))

## Where to go next

- **Hierarchical ToC.** For longer documents, summarize *sections* (groups of pages) on top of page summaries. Retrieval then walks the tree top-down — first pick the section, then the pages — instead of scanning every page summary.
- **Multi-document corpora.** Add a per-document description and a routing step that picks the right document before picking pages.
- **Prompt caching.** The ToC is reused across every query — wrap it in an Anthropic `cache_control` block to cut retrieval cost dramatically.
- **Compare against vector RAG.** Run both on the same questions and look at: answer quality, latency, cost per query, and how often you can trace *why* a chunk was retrieved.